# En este ipynb se muestra la creación de la ontología usada de animes.

In [2]:
import sbc_tools as sbc
from SPARQLWrapper import SPARQLWrapper, JSON
from rdflib import Graph, Namespace, URIRef, Literal
from rdflib.namespace import RDF, RDFS, OWL

In [3]:
grafo = Graph()
grafo = sbc.load("anime_knowledge_graph_clean.ttl")

sbc.print_class_hierarchy(grafo)


JERARQUÍA DE CLASES DE LA ONTOLOGÍA
📁 Genre (Genre)
  └─ ASMR: ASMR
  └─ Action: Sin etiqueta
  └─ Adventure: Sin etiqueta
  └─ Avant_Garde: Sin etiqueta
  └─ Award_Winning: Sin etiqueta
  └─ BL: BL
  └─ Boys'_Love_game: Boys' Love game
  └─ Comedy: Sin etiqueta
  └─ Drama: Sin etiqueta
  └─ Ecchi: Sin etiqueta
  └─ Fantasy: Sin etiqueta
  └─ Girls_Love: Sin etiqueta
  └─ Gourmet: Sin etiqueta
  └─ Horror: Sin etiqueta
  └─ Japanese_idol: Japanese idol
  └─ Jidaigeki: Jidaigeki
  └─ LGBT-related_television_series: LGBT-related television series
  └─ LGBT-themed_videogame: LGBT-themed video game
  └─ Mystery: Sin etiqueta
  └─ Romance: Sin etiqueta
  └─ SciFi: Sin etiqueta
  └─ Sentai: Sentai
  └─ Sports: Sin etiqueta
  └─ Supernatural: Sin etiqueta
  └─ Suspense: Sin etiqueta
  └─ Westerns_on_television: Westerns on television
  └─ Xianxia: Xianxia
  └─ action-adventure_game: action-adventure game
  └─ action_anime_and_manga: action anime and manga
  └─ action_comics: action comics
  

In [ ]:
from rdflib import Graph, Namespace, Literal, RDF, XSD, URIRef
import requests
import re
import time

def clean_uri(name):
    name = name.strip()
    name = re.sub(r'\s+', '_', name)
    name = re.sub(r'[^a-zA-Z0-9_]', '', name)
    return name

max_consecutive_errors = 50
errors = 0

# Cargar grafo existente
g = sbc.load("grafo_fusionado.ttl", format="turtle")

# Namespaces
ONTO = Namespace("http://www.fdi.ucm.es/mia/sbc/animeontology/")
NS2 = Namespace("http://www.semanticweb.org/eladi/ontologies/anime-ontology#")
anime_id = 1
while True:
    
    try:
        anime_data = requests.get(f"https://api.jikan.moe/v4/anime/{anime_id}").json()['data']
    except:
        print(f"Anime ID {anime_id} error")
        errors += 1
        anime_id += 1

        if errors >= max_consecutive_errors:
            print("Probablemente no quedan más animes válidos. Deteniendo.")
            break

        continue



    if errors >= max_consecutive_errors:
        print("Probablemente no quedan más animes válidos. Deteniendo.")
        break


    errors = 0 
    title = anime_data.get('title_english') or anime_data['title']
    anime_uri = ONTO[clean_uri(title)]

    # AnimeSeries
    if (anime_uri, RDF.type, ONTO.AnimeSeries) not in g:
        g.add((anime_uri, RDF.type, ONTO.AnimeSeries))

    # Títulos
    for key, value in [('englishTitle', anime_data.get('title_english')), ('episodeTitle', anime_data['title'])]:
        if value:
            pred = getattr(NS2, key)
            lit = Literal(value, datatype=XSD.string)
            if (anime_uri, pred, lit) not in g:
                g.add((anime_uri, pred, lit))

    # Número de episodios y fecha
    if anime_data.get('episodes'):
        lit = Literal(anime_data['episodes'], datatype=XSD.integer)
        if (anime_uri, ONTO.numEpisodes, lit) not in g:
            g.add((anime_uri, ONTO.numEpisodes, lit))
    if anime_data.get('aired') and anime_data['aired'].get('from'):
        lit = Literal(anime_data['aired']['from'], datatype=XSD.dateTime)
        if (anime_uri, ONTO.startDate, lit) not in g:
            g.add((anime_uri, ONTO.startDate, lit))

    # Géneros
    for genre in anime_data.get('genres', []):
        genre_uri = ONTO[clean_uri(genre['name'])]
        if (genre_uri, RDF.type, ONTO.Genre) not in g:
            g.add((genre_uri, RDF.type, ONTO.Genre))
            g.add((genre_uri, NS2.label, Literal(genre['name'], datatype=XSD.string)))
        if (anime_uri, ONTO.hasGenre, genre_uri) not in g:
            g.add((anime_uri, ONTO.hasGenre, genre_uri))

    # Estudios
    for studio in anime_data.get('studios', []):
        studio_uri = ONTO[clean_uri(studio['name'])]
        if (studio_uri, RDF.type, NS2.Studio) not in g:
            g.add((studio_uri, RDF.type, NS2.Studio))
            g.add((studio_uri, NS2.label, Literal(studio['name'], datatype=XSD.string)))
        if (studio_uri, NS2.produced, anime_uri) not in g:
            g.add((studio_uri, NS2.produced, anime_uri))

    # === Staff (directores, productores, etc.) ===
    try:
        staff_data = requests.get(f"https://api.jikan.moe/v4/anime/{anime_id}/staff").json()['data']
        for s in staff_data:
            staff_uri = ONTO[clean_uri(s['person']['name'])]
            if (staff_uri, RDF.type, NS2.RealPerson) not in g:
                g.add((staff_uri, RDF.type, NS2.RealPerson))
                g.add((staff_uri, NS2.label, Literal(s['person']['name'], datatype=XSD.string)))
            # Cada posición como rol de producción
            for pos in s.get('positions', []):
                role_uri = ONTO[clean_uri(pos)]
                if (role_uri, RDF.type, NS2.ProductionRole) not in g:
                    g.add((role_uri, RDF.type, NS2.ProductionRole))
                    g.add((role_uri, NS2.label, Literal(pos, datatype=XSD.string)))
                if (role_uri, NS2.roleIn, anime_uri) not in g:
                    g.add((role_uri, NS2.roleIn, anime_uri))
    except:
        pass

    # === Characters y VoiceActors ===
    try:
        char_data = requests.get(f"https://api.jikan.moe/v4/anime/{anime_id}/characters").json()['data']
        for c in char_data:
            char_uri = ONTO[clean_uri(c['character']['name'])]
            if (char_uri, RDF.type, NS2.Character) not in g:
                g.add((char_uri, RDF.type, NS2.Character))
                g.add((char_uri, NS2.label, Literal(c['character']['name'], datatype=XSD.string)))
            if (c['role'].lower() == 'main'):
                if (char_uri, RDF.type, NS2.MainCharacter) not in g:
                    g.add((char_uri, RDF.type, NS2.MainCharacter))
            if (char_uri, NS2.characterIn, anime_uri) not in g:
                g.add((char_uri, NS2.characterIn, anime_uri))
            # Voice actors
            for va in c.get('voice_actors', []):
                if va['language'].lower() != 'japanese':
                    continue
                va_uri = ONTO[clean_uri(va['person']['name'])]
                if (va_uri, RDF.type, NS2.VoiceActor) not in g:
                    g.add((va_uri, RDF.type, NS2.VoiceActor))
                    g.add((va_uri, NS2.label, Literal(va['person']['name'], datatype=XSD.string)))
                if (char_uri, NS2.dubbedBy, va_uri) not in g:
                    g.add((char_uri, NS2.dubbedBy, va_uri))
    except:
        pass

    # === Episodios ===
    try:
        eps_data = requests.get(f"https://api.jikan.moe/v4/anime/{anime_id}/episodes").json()['data']
        for ep in eps_data:
            ep_uri = ONTO[clean_uri(ep['title'])]
            if (ep_uri, RDF.type, NS2.Movie) not in g:
                g.add((ep_uri, RDF.type, NS2.Episode))  # o ns2:Episode según tu ontología
            if (ep_uri, NS2.episodeTitle, Literal(ep['title'], datatype=XSD.string)) not in g:
                g.add((ep_uri, NS2.episodeTitle, Literal(ep['title'], datatype=XSD.string)))
            if (ep_uri, NS2.isEpisodeOf, anime_uri) not in g:
                g.add((ep_uri, NS2.isEpisodeOf, anime_uri))
    except:
        pass

    print(f"Procesado: {title} (ID {anime_id})")
    time.sleep(0.5)

    anime_id += 1

  

# Guardar grafo actualizado
g.serialize("grafo_actualizado_todo.ttl", format="turtle")
print("¡Grafo completamente enriquecido listo!")
sbc.show_graph(g, output_file="grafo_actualizado_completo.html")


Procesado: Cowboy Bebop (ID 1)
Anime ID 2 error
Anime ID 3 error
Anime ID 4 error
Procesado: Cowboy Bebop: The Movie (ID 5)
Procesado: Trigun (ID 6)
Procesado: Witch Hunter Robin (ID 7)
Procesado: Beet the Vandel Buster (ID 8)
Anime ID 9 error
Anime ID 10 error
Anime ID 11 error
Anime ID 12 error
Anime ID 13 error
Anime ID 14 error
Anime ID 15 error
Procesado: Honey and Clover (ID 16)
Procesado: Hungry Heart: Wild Striker (ID 17)
Procesado: Initial D Fourth Stage (ID 18)
Procesado: Monster (ID 19)
Procesado: Naruto (ID 20)
Procesado: One Piece (ID 21)
Procesado: The Prince of Tennis (ID 22)
Procesado: Ring ni Kakero 1 (ID 23)
Procesado: School Rumble (ID 24)
Procesado: Desert Punk (ID 25)
Procesado: Texhnolyze (ID 26)
Procesado: Trinity Blood (ID 27)
Procesado: Yakitate!! Japan (ID 28)
Procesado: Zipang (ID 29)
Procesado: Neon Genesis Evangelion (ID 30)
Procesado: Neon Genesis Evangelion: Death & Rebirth (ID 31)
Procesado: Neon Genesis Evangelion: The End of Evangelion (ID 32)
Procesad

In [4]:
sbc.print_class_hierarchy(g)

NameError: name 'g' is not defined

In [7]:
# como vemos, han quedado algunas clases basura, producto de los cambios realizados, eliminanos "hasGenre" y todo lo que sea a onto:Genre
onto = Namespace("http://www.fdi.ucm.es/mia/sbc/animeontology/")
ns1 = Namespace("http://www.semanticweb.org/eladi/ontologies/anime-ontology#")

generos = set(grafo.subjects(RDF.type, onto.Genre))

print("Géneros encontrados:", len(generos))

for genre in generos:
    for p, o in list(grafo.predicate_objects(genre)):
        grafo.remove((genre, p, o))
    
    for s, p in list(grafo.subject_predicates(genre)):
        grafo.remove((s, p, genre))
    
    grafo.remove((genre, RDF.type, onto.Genre))

grafo.remove((onto.Genre, RDF.type, OWL.Class))

# eliminamos tambien la propiedad hasGenre
props = [
    onto.hasGenre,
    ns1.hasGenre
]

for prop in props:
    for p, o in list(grafo.predicate_objects(prop)):
        grafo.remove((prop, p, o))
    
    for s, p in list(grafo.subject_predicates(prop)):
        grafo.remove((s, p, prop))

#eliminanos la propiedad hasgenre de animeseries
grafo.remove((onto.hasGenre, RDF.type, OWL.ObjectProperty))
sbc.print_class_hierarchy(grafo)

sbc.save(grafo, "ontologiaAnimes.ttl")



Géneros encontrados: 0

JERARQUÍA DE CLASES DE LA ONTOLOGÍA
📁 Studio (anime-ontology#Studio) [anime-ontology#produced]
  └─ ACGT: Sin etiqueta
  └─ AIC: Sin etiqueta
  └─ Bee_Train: Sin etiqueta
  └─ Bones: Sin etiqueta
  └─ Daume: Sin etiqueta
  └─ Digital_Frontier: Sin etiqueta
  └─ Gainax: Sin etiqueta
  └─ Gallop: Sin etiqueta
  └─ Gonzo: Sin etiqueta
  └─ Group_TAC: Sin etiqueta
  └─ HAL_Film_Maker: Sin etiqueta
  └─ JCStaff: Sin etiqueta
  └─ Kyoto_Animation: Sin etiqueta
  └─ Madhouse: Sin etiqueta
  └─ Nippon_Animation: Sin etiqueta
  └─ Nomad: Sin etiqueta
  └─ OLM: Sin etiqueta
  └─ Production_IG: Sin etiqueta
  └─ Seven_Arcs: Sin etiqueta
  └─ Studio_Comet: Sin etiqueta
  └─ Studio_Deen: Sin etiqueta
  └─ Studio_Pierrot: Sin etiqueta
  └─ Sunrise: Sin etiqueta
  └─ Tatsunoko_Production: Sin etiqueta
  └─ Toei_Animation: Sin etiqueta
  └─ Tokyo_Movie_Shinsha: Sin etiqueta
  └─ Trans_Arts: Sin etiqueta
  └─ Xebec: Sin etiqueta
  └─ Zexcs: Sin etiqueta
  └─ asread: Sin etiqueta